### plot the timeseries of the wq parameters and store them in a pdf

In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# -------------------------------
# Input & Output Paths
# -------------------------------
input_folder = r"H:\datagaps\testdata"
output_pdf = r"H:\datagaps\STREAM_WQ_Timeseries_AllStations.pdf"

# H:\Lab_System\datacube\data\raw\STREAM\STREAM-Mississippi\02_waterquality_timeseries

# Expected water quality parameters
wq_parameters = [
    "WTemp_C", "SpC_uScm", "DO_mgL", "pH",
    "Turb_FNU", "Turb_NTU", "NO3_mgNL",
    "fDOM_QSU", "fDOM_RFU", "DOC_mgL",
    "PO4_mgL", "Chla_ugL", "Chla_RFU",
    "PC_ugL", "PC_RFU"
]

# Get all CSV files
csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(output_pdf), exist_ok=True)

with PdfPages(output_pdf) as pdf:
    
    for file in csv_files:
        try:
            # Read CSV
            df = pd.read_csv(file)
            
            # Skip if DateTime not present
            if "DateTime" not in df.columns:
                continue
            
            # Convert DateTime
            df["DateTime"] = pd.to_datetime(df["DateTime"], errors='coerce')
            df = df.sort_values("DateTime")
            
            # Keep only available WQ parameters
            available_params = [col for col in wq_parameters if col in df.columns]
            
            if not available_params:
                continue
            
            # Create figure
            n_params = len(available_params)
            fig, axes = plt.subplots(n_params, 1, figsize=(11, 2.5*n_params), sharex=True)
            
            if n_params == 1:
                axes = [axes]
            
            for ax, param in zip(axes, available_params):
                ax.plot(df["DateTime"], df[param])
                ax.set_ylabel(param)
                ax.grid(True)
            
            # Gauge station name from filename
            gauge_name = os.path.splitext(os.path.basename(file))[0]
            
            fig.suptitle(f"Water Quality Time Series\n{gauge_name}", fontsize=14)
            plt.xlabel("DateTime")
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            
            pdf.savefig(fig)
            plt.close(fig)
            
            print(f"Processed: {gauge_name}")
        
        except Exception as e:
            print(f"Error processing {file}: {e}")

print(f"\nPDF successfully saved at:\n{output_pdf}")

Processed: STREAM-gauge-16
Processed: STREAM-gauge-17
Processed: STREAM-gauge-18
Processed: STREAM-gauge-19
Processed: STREAM-gauge-20
Processed: STREAM-gauge-21
Processed: STREAM-gauge-22
Processed: STREAM-gauge-23


C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (12) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Processed: STREAM-gauge-24
Processed: STREAM-gauge-25
Processed: STREAM-gauge-26


C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (6,10) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Processed: STREAM-gauge-27
Processed: STREAM-gauge-28
Processed: STREAM-gauge-29
Processed: STREAM-gauge-30

PDF successfully saved at:
H:\datagaps\STREAM_WQ_Timeseries_AllStations.pdf


#### Answer the following questions based on the timeseries data:
1) Percentage of Data missing between the start and end date.
  
2) The length between start and end datetime (hour) 

3) longest continous recored between the start and end date.  

In [5]:
import os
import pandas as pd
import numpy as np

# Folder path
folder_path = r"H:\Lab_System\datacube\data\raw\STREAM\STREAM-Mississippi\02_waterquality_timeseries"

# List of possible parameters
parameters = [
    "WTemp_C", "SpC_uScm", "DO_mgL", "pH",
    "Turb_FNU", "Turb_NTU", "NO3_mgNL",
    "fDOM_QSU", "fDOM_RFU", "DOC_mgL",
    "PO4_mgL", "Chla_ugL", "Chla_RFU",
    "PC_ugL", "PC_RFU"
]

# Store all results
results = []

# Loop through each CSV
for file in os.listdir(folder_path):
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(folder_path, file)
    
    # Extract STREAM_ID from filename (remove .csv)
    stream_id = os.path.splitext(file)[0]

    df = pd.read_csv(file_path)

    if "DateTime" not in df.columns:
        continue

    # Convert DateTime
    df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
    df = df.dropna(subset=["DateTime"])
    df = df.sort_values("DateTime")

    # Create full hourly range
    start = df["DateTime"].min()
    end = df["DateTime"].max()
    full_range = pd.date_range(start=start, end=end, freq="H")

    # Reindex to full hourly timeline
    df = df.set_index("DateTime").reindex(full_range)

    # Total expected length in hours
    total_hours = len(full_range)

    station_result = {"STREAM_ID": stream_id}

    for param in parameters:
        if param in df.columns:

            series = df[param]

            # 1 Percentage Missing
            missing_count = series.isna().sum()
            percent_missing = (missing_count / total_hours) * 100

            # 2 Length between start and end
            length_hours = total_hours

            # 3 Longest continuous record (non-missing streak)
            not_na = series.notna().astype(int)
            groups = (not_na.diff() != 0).cumsum()
            streak_lengths = not_na.groupby(groups).sum()
            longest_streak = streak_lengths.max() if len(streak_lengths) > 0 else 0

            station_result[f"{param}_pct_missing"] = round(percent_missing, 2)
            station_result[f"{param}_length_hr"] = length_hours
            station_result[f"{param}_longest_hr"] = int(longest_streak)

        else:
            # If parameter not present
            station_result[f"{param}_pct_missing"] = np.nan
            station_result[f"{param}_length_hr"] = np.nan
            station_result[f"{param}_longest_hr"] = np.nan

    results.append(station_result)

# Convert to dataframe
summary_df = pd.DataFrame(results)

# Save output
summary_df.to_csv(r'H:\datagaps\temporal_statistics.csv', index=False)

print("Processing complete.")

C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (4,6,8,10) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (12,14,16) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (12) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (14,16) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactiv

C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (6,10) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
C:\Users\kuntl\anaconda3\envs\sai\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (12) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Processing complete.


In [6]:
import pandas as pd

In [7]:
db = pd.read_csv(r'H:\Lab_System\datacube\data\raw\STREAM\STREAM-Mississippi\01_metadata\metadata.csv')
db

,STREAM_ID,sourceID,source,site name,latitude_wgs84,longitude_wgs84,drainagearea_sqkm,state_name,time_zone,WQ_parameters
0,STREAM-gauge-1723,3007800,USGS,"Allegheny River at Port Allegany, PA",41.818676,-78.292791,642.31752,Pennsylvania,Eastern,"WTemp_C,SpC_uScm,DO_mgL,pH"
1,STREAM-gauge-2691,3011020,USGS,ALLEGHENY RIVER AT SALAMANCA NY,42.156833,-78.715556,4164.70392,New York,Eastern,WTemp_C
2,STREAM-gauge-1724,3012545,USGS,"Allegheny River below Kinzua Dam at Big Bend, PA",41.838672,-79.004925,5646.17820,Pennsylvania,Eastern,"WTemp_C,DO_mgL"
3,STREAM-gauge-1725,3012550,USGS,"Allegheny River at Kinzua Dam, PA",41.841449,-79.011985,5646.17820,Pennsylvania,Eastern,"WTemp_C,DO_mgL"
4,STREAM-gauge-1726,3016000,USGS,"Allegheny River at West Hickory, PA",41.570895,-79.407824,9479.36340,Pennsylvania,Eastern,"WTemp_C,SpC_uScm"
...,...,...,...,...,...,...,...,...,...,...
865,STREAM-gauge-3060,460614111234801,USGS,Missouri River above Sixteen Mile Creek nr Lom...,46.103818,-111.397463,NaN,Montana,Mountain,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU"
866,STREAM-gauge-3061,460709111241001,USGS,Missouri River bl Toston Dam nr Toston,46.119235,-111.403686,NaN,Montana,Mountain,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU"
867,STREAM-gauge-3062,461411111280601,USGS,Missouri River ab York Island nr Toston,46.236346,-111.469217,NaN,Montana,Mountain,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU"
868,STREAM-gauge-3063,462107111312301,USGS,Missouri River ab Canyon Ferry nr Townsend,46.351876,-111.523858,NaN,Montana,Mountain,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU"


In [9]:
db['drainagearea_sqkm'].isna().sum()

122